In [27]:
import os
import json

for filename in os.listdir('./items/'):
    if not filename.endswith('.json'):
        continue
    item = json.load(open(f'./items/{filename}'))
    if 'knowledge_components' not in item['classification']:
        continue
    # merge knowledge components that starts wit 'sql concept ...' into a single knowledge component 'sql concept'
    sql_todel = set()
    sql_concepts = set()
    sql_notes = set()
    for kc, values in item['classification']['knowledge_components'].items():
        if kc.startswith('sql concept'):
            sql_notes.add(kc.replace('sql concept', '').replace('(', '').replace(')', '').strip())
            sql_concepts.update(values['concepts'])
            sql_notes.add(values['note'])
            sql_todel.add(kc)
    
    if len(sql_concepts) < 1:
        continue
    
    for kc in sql_todel:
        del item['classification']['knowledge_components'][kc]
    item['classification']['knowledge_components']['sql concept'] = {
        'concepts': list(filter(lambda x: len(x.strip()) > 0, sql_concepts)),
        'note': ','.join(filter(lambda x: len(x.strip()) > 0, sql_notes)),
        "created_at": None,
        "updated_at": None
    }
    open(f'./items/{filename}', 'w').write(json.dumps(item, indent=2))